In [ ]:
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from nltk.tokenize import word_tokenize
from tensorflow.keras.models import load_model # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
import pickle
import numpy as np

#caminhos 
path_models = '../models/'
path_ia = path_models + 'chatbotIA.keras'
path_tokenizer = path_models + 'tokenizer.pkl'
path_label_encoder = path_models + 'label_encoder.pkl'

#modelo já treinado
model = load_model(path_ia)

# Carregar o tokenizer (o "dicionário" de palavras)
with open(path_tokenizer, 'rb') as f:
    tokenizer = pickle.load(f)

# Carregar o label encoder (o "tradutor" de IDs para respostas)
with open(path_label_encoder, 'rb') as f:
    label_encoder = pickle.load(f)

#Nova função de responder, usando o modelo ja treinado
def responder(mensagem): #função para carregar e abrir o arquivo

    print(f"Mensagem recebida: {mensagem}")

    # Pré-processar a mensagem do usuário, isso transforma a frase (ex: "oi") em uma sequência de IDs (ex: [2]) e finalmente posso deletar uma função feito para isso
    seq = tokenizer.texts_to_sequences([mensagem])

    # Padronizar a sequência (ex: [2]) em um vetor de tamanho 'max_len' (ex: [2, 0, 0, ...])
    padded_seq = pad_sequences(seq, maxlen=20, padding='post')

    # Solicitar a previsão do modelo retorna um array de probabilidades (ex: [0.1, 0.05, 0.8, 0.05])
    predicao = model.predict(padded_seq)

    # Tentar encontrar a melhor resposta np.argmax encontra o índice (ID) da maior probabilidade (ex: 2)
    resposta_id = np.argmax(predicao)

    # "Destraduzir" a resposta, basicamente a label_encoder transforma o ID (ex: 2) de volta em texto (ex: "Desculpe, não entendi.")
    resposta_texto = label_encoder.inverse_transform([resposta_id])
    
    # Retorna o texto da resposta, mas ainda esta tudo genérico e sem base de dados abrangente
    return resposta_texto[0]

    '''
    codigo legado abaixo, que usava pandas para buscar na base de dados CSV, precisa usar como base de apoio
    base_dados = '../data/iniciacao.csv' #Definir o nome do arquivo CSV (depois iremos unificar tudo em um mesmo arquivo)

    #Try é uma tentativa do sistema
    try:
        
        df = pd.read_csv(base_dados) #carregar o arquivo solicitado sendo o iniciacao e cria um dataframe
    
         É apenas para verificar que o dataframe foi carregado com sucesso e printar
        print("--- DataFrame (CSV) Carregado ---")
        print(df)
        print("---------------------------------\n")
        
        duvida = pre_processamento(mensagem) #Recebe a mensagem toketizada

        # Vai procurar apenas a coluna perguntas de acordo 
        # Usamos 'str.contains()' para verificar se a entrada está *dentro* do texto da coluna.
        # 'case=False' ignora diferenças entre maiúsculas e minúsculas (ex: "Oi" ou "oi").
        # 'na=False' trata células vazias (NaN) como se não tivessem a palavra.
        perguntas = df['perguntas'].str.contains(duvida, case=False, na=False)
    
        # Filtrar o DataFrame para obter apenas as linhas que correspondem
        resposta_df = df[perguntas]

        # Verificar se esta vazio para retornar a resposta ao usuario
        if not resposta_df.empty:

            # Pega a resposta da *primeira* correspondência encontrada
            response = resposta_df.iloc[0]['respostas']

            #Prints apenas para verificar a entrada e saida de dados
            #print(f"Entrada: '{duvida}'")
            #print(f"Resposta: {response}")
            return response
        else:
            # Se o DataFrame 'resposta_df' estiver vazio (nenhuma correspondência)
            print(f"Entrada: '{duvida}'")
            print("Resposta: Desculpe, não encontrei uma resposta para isso.")
        
    #Caso alguém mexa na base de dados 
    except FileNotFoundError:
        print(f"Erro: O arquivo '{base_dados}' não foi encontrado.")
        print("Por favor, verifique se a base de dados está na mesma pasta do seu notebook.")
    '''

2025-11-17 22:02:35.342112: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-17 22:02:54.840040: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-17 22:03:06.101354: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-17 22:03:10.168994: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
print("ChatBot: Olá! Digite 'sair' para encerrar.\n")

while True:
    usuario = input("Você: ")
    if usuario.lower() == "sair":
        print("ChatBot: Até mais!")
        break
    
    # Chamar nova função de IA, obs.: tem um ignore, pois vive apresentando alerta a qualquer modificação na função
    resposta = responder(usuario) # type: ignore
    print("ChatBot (IA):", resposta)


ChatBot: Olá! Digite 'sair' para encerrar.

Mensagem recebida: oi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step
ChatBot (IA): Oi! Tudo bem?
Mensagem recebida: olá
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
ChatBot (IA): Sinto muito em ouvir isso. Como posso ajudar?
ChatBot: Até mais!


'\nwhile True:\n    usuario = input("Você: ")\n    if usuario.lower() == "sair":\n        print("ChatBot: Até mais!")\n        break\n    print("ChatBot:", responder(usuario))\n'